In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

# 1. Kupakua data
url = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
df = pd.read_csv(url)

# 2. Kujaza zeros sehemu zenye upungufu
df = df.fillna(0)

# 3. Kugawanya data (60% train, 20% validation, 20% test)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

# 4. DictVectorizer
train_dicts = df_train.to_dict(orient='records')
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)
print("Data imesomwa na kukamilishwa vizuri!")

Data imesomwa na kukamilishwa vizuri!


In [3]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import export_text

dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- vehicle_weight <= 3022.11
|   |--- value: [16.88]
|--- vehicle_weight >  3022.11
|   |--- value: [12.94]



In [11]:
from sklearn.metrics import mean_squared_error

for n in range(10, 201, 10):
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print("n_estimators=", n, "RMSE=", round(rmse, 4))

n_estimators= 10 RMSE= 0.4596
n_estimators= 20 RMSE= 0.4539
n_estimators= 30 RMSE= 0.4522
n_estimators= 40 RMSE= 0.4491
n_estimators= 50 RMSE= 0.4473
n_estimators= 60 RMSE= 0.4459
n_estimators= 70 RMSE= 0.4457
n_estimators= 80 RMSE= 0.4456
n_estimators= 90 RMSE= 0.4453
n_estimators= 100 RMSE= 0.4451
n_estimators= 110 RMSE= 0.444
n_estimators= 120 RMSE= 0.4443
n_estimators= 130 RMSE= 0.4439
n_estimators= 140 RMSE= 0.4436
n_estimators= 150 RMSE= 0.4432
n_estimators= 160 RMSE= 0.4429
n_estimators= 170 RMSE= 0.4429
n_estimators= 180 RMSE= 0.4425
n_estimators= 190 RMSE= 0.4426
n_estimators= 200 RMSE= 0.4425


In [13]:
from sklearn.metrics import mean_squared_error

for d in [10, 15, 20, 25]:
    rmses = []
    for n in range(10, 201, 10):
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmses.append(rmse)
    print("max_depth=", d, "Mean RMSE=", round(np.mean(rmses), 4))

max_depth= 10 Mean RMSE= 0.4419
max_depth= 15 Mean RMSE= 0.4451
max_depth= 20 Mean RMSE= 0.4462
max_depth= 25 Mean RMSE= 0.4459


In [14]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

for f, imp in zip(dv.get_feature_names_out(), rf.feature_importances_):
    print(f"{f}: {imp}")

acceleration: 0.011499190803846305
drivetrain=All-wheel drive: 0.0003186782725673308
drivetrain=Front-wheel drive: 0.0003742124217381939
engine_displacement: 0.0032979109412370395
fuel_type=Diesel: 0.00035499078855189
fuel_type=Gasoline: 0.0003316505069333763
horsepower: 0.016006642864127722
model_year: 0.0032244074424087827
num_cylinders: 0.002308389682358528
num_doors: 0.0016272155115303305
origin=Asia: 0.00047020043700694906
origin=Europe: 0.0005315071064835319
origin=USA: 0.0005407204629783297
vehicle_weight: 0.9591142827582317


In [1]:
import xgboost as xgb
import re

# Kusafisha majina ya columns kuwa salama kwa DMatrix
df_train_xgb = pd.DataFrame(X_train.toarray(), columns=list(dv.get_feature_names_out()))
df_train_xgb.columns = [re.sub(r'[\[\]<>]', '', col) for col in df_train_xgb.columns]

df_val_xgb = pd.DataFrame(X_val.toarray(), columns=list(dv.get_feature_names_out()))
df_val_xgb.columns = [re.sub(r'[\[\]<>]', '', col) for col in df_val_xgb.columns]

features_clean = list(df_train_xgb.columns)

dtrain = xgb.DMatrix(df_train_xgb, label=y_train, feature_names=features_clean)
dval = xgb.DMatrix(df_val_xgb, label=y_val, feature_names=features_clean)

watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta_val in [0.3, 0.1]:
    params = {
        'eta': eta_val,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 0,
    }
    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    print(f"Imejaribu eta={eta_val} imekamilika!")

ValueError: source code string cannot contain null bytes